### Performance evaluations: Precision - Recall and Average Percision ###

In [1]:
import sys
import os
import numpy as np
import json
import copy
import glob
import pandas as pd
import logging
from pathlib import Path
from matplotlib import pyplot as plt
from matplotlib.patches import Rectangle
from sklearn import metrics
import seaborn as sns

logger = logging.getLogger(__name__)

# PyTorch
import torch
from torchvision import ops

%load_ext autoreload
%autoreload 2
import computervision
from computervision.imageproc import is_image, ImageData, clipxywh, xyxy2xywh, xywh2xyxy, plot_boxes
from computervision.datasets import DETRdataset
from computervision.transformations import AugmentationTransform
from computervision.performance import DetectionMetrics
from computervision.inference import DETRinference, get_gpu_info

print(f'Project version: {computervision.__version__}')
print(f'Authors: {computervision.__authors__}')
print(f'Python version:  {sys.version}')

Project version: v0.0.2
Authors: The Core for Computational Biomedicine at Harvard Medical School
https://dbmi.hms.harvard.edu/about-dbmi/core-computational-biomedicine
Python version:  3.12.3 (main, Jun 18 2025, 17:59:45) [GCC 13.3.0]


In [10]:
# Check GPU availability
device, device_str = get_gpu_info()

CUDA available: True
Number of GPUs found:  1
Current device ID: 0
GPU device name:   NVIDIA GeForce RTX 3060 Laptop GPU
PyTorch version:   2.8.0a0+34c6371d24.nv25.08
CUDA version:      13.0
CUDNN version:     91200
Current device:    cuda:0


### Test data ###

In [11]:
# Dentex test data
data_dir = os.environ.get('DATA')
dataset_name = 'dataset_object_dentex_251011'
image_dir = os.path.join(data_dir, dataset_name, 'test')
test_df_file_name = 'dataset_object_dentex_251011_test.parquet'
test_df_file = os.path.join(image_dir, test_df_file_name)
df = pd.read_parquet(test_df_file)

# Filter the data frame
dset_col = 'dset'
pos_col = 'ada'
file_col = 'file_name'
bbox_col = 'bbox'
df = df.loc[(df[dset_col] == 'test') & (df['transformation'] == 5)]
display(df.head(2))
print(f'Images in test data: {len(df[file_col].unique())}')
print(f'Annotations:         {df.shape[0]}')

,bbox,quadrant,ada,file_name,file_base_name,quadrants,height,width,transformation,transformation_name,dset
10256,"[572, 228, 67, 338]",1,8,test_train_219_01_05.png,train_219,1,640,640,5,test_set,test
10257,"[511, 234, 91, 332]",1,7,test_train_219_01_05.png,train_219,1,640,640,5,test_set,test


Images in test data: 160
Annotations:         1275


### Model ###

In [12]:
model_name = 'rtdetr_dtx_251012_08'
model_dir = os.path.join(data_dir, 'model', model_name)

checkpoint_paths = glob.glob(os.path.join(model_dir, 'checkpoint-*'))
checkpoint_numbers = [int(os.path.basename(checkpoint).split('-')[-1]) for checkpoint in checkpoint_paths]
checkpoints = dict(zip(checkpoint_numbers, checkpoint_paths))

# checkpoint_dir = os.path.join(model_dir, f'checkpoint-{checkpoint}')
display(checkpoints)

model_config_file = os.path.join(model_dir, f'{model_name}.json')
with open(model_config_file, mode='r') as file:
    model_config = json.load(file)
display(*list(model_config.keys()), sep='\n')

# Image processor
processor = DETRinference(device_name='cuda:0', 
                          checkpoint_path=checkpoint_paths[0]).processor

{19600: '/app/data_model/model/rtdetr_dtx_251012_08/checkpoint-19600',
 119900: '/app/data_model/model/rtdetr_dtx_251012_08/checkpoint-119900',
 120000: '/app/data_model/model/rtdetr_dtx_251012_08/checkpoint-120000',
 119950: '/app/data_model/model/rtdetr_dtx_251012_08/checkpoint-119950',
 119850: '/app/data_model/model/rtdetr_dtx_251012_08/checkpoint-119850'}

'model_info'

'id2label'

'training_args'

'processor_params'

'bbox_format'

### PyTorch Dataset ###

In [14]:
# Create a PyTorch data set
transforms = AugmentationTransform().get_transforms(name='val')
bbox_format = model_config.get('bbox_format')
dataset = DETRdataset(data=df.copy(), 
                      image_processor=processor, 
                      image_dir=image_dir, 
                      file_name_col=file_col, 
                      label_id_col=None, 
                      bbox_col=None, 
                      bbox_format=bbox_format, 
                      transforms=transforms)
print(f'Total images in test data: {len(dataset)}')

Total images in test data: 160


### Predict bounding boxes on the test data for all checkpoints from this model ###

In [26]:
# Run the forward pass to get the predictions for all checkpoints
threshold = 0.05
pred_raw_list = []
for c, (checkpoint, checkpoint_path) in enumerate(checkpoints.items()):
    print(f'Running checkpoint {checkpoint} {c + 1} / {len(checkpoints)}')
    dtr = DETRinference(device_name='cuda:0', 
                        checkpoint_path=checkpoint_path,
                        batch_size=16)
    pred = dtr.predict_on_dataset(dataset, threshold=threshold)
    pred = pred.assign(checkpoint=checkpoint, pred_threshold=threshold)
    pred_raw_list.append(pred)
    pred_raw = pd.concat(pred_raw_list, axis=0, ignore_index=True).\
                    sort_values(by='checkpoint', ascending=True).\
                    reset_index(drop=True)

Running checkpoint 19600 1 / 5
Predicting batch 10 of 10.
Running checkpoint 119900 2 / 5
Predicting batch 10 of 10.
Running checkpoint 120000 3 / 5
Predicting batch 10 of 10.
Running checkpoint 119950 4 / 5
Predicting batch 10 of 10.
Running checkpoint 119850 5 / 5
Predicting batch 10 of 10.


In [33]:
# Make a copy of the predictions 
pred = copy.deepcopy(pred_raw)

# Add the file names to the data frame
file_names = df[file_col].unique()
id2file = dict(zip(range(len(file_names)), file_names))
pred[file_col] = pred['image_id'].apply(lambda image_id: id2file.get(image_id))

# Add the label names (the positions) to the data frame
id2label = {int(category_id): int(label) for category_id, label in model_config.get('id2label').items()}
pred[pos_col] = pred['category_id'].apply(lambda category_id: id2label.get(category_id))

print(f'Images in output data:     {len(pred['image_id'].unique())}')

# Let's find out if all of the images resulted in predictions
pred_empty = len(pred.loc[pred[pos_col].isnull(), file_col].unique())

# Filter out rows with images that do not have predictions
pred = pred.loc[~pred[pos_col].isnull()]
print(f'Total number of images in data set:             {len(dataset)}')
print(f'Images without predictions for threshold ({threshold}):{pred_empty}')
print(f'Images with predictions:                        {len(pred[file_col].unique())}')

Images in output data:     160
Total number of images in data set:             160
Images without predictions for threshold (0.05):0
Images with predictions:                        160


### Classify predictions: TP, FP, FN ###
We create a method to classify all of the predictions in the data set. 

In [42]:
true_df = copy.deepcopy(df)
pred_df = copy.deepcopy(pred)
bbox_col = 'bbox'
label_col = 'ada'
file_col = 'file_name'
score_col = 'score'
iou_threshold = 0.5
display(pred_df.head(2))

# Save the output
output_dir = os.path.join(os.environ['DATA'], 'output')
Path(output_dir).mkdir(parents=True, exist_ok=True)
print(output_dir)

,image_id,image_width,image_height,batch,category_id,bbox,score,area,checkpoint,pred_threshold,file_name,ada
0,0,640,640,0,5,"[419, 159, 119, 407]",0.289987,48433,19600,0.05,test_train_219_01_05.png,6
1,105,640,640,6,14,"[416, 183, 74, 209]",0.109004,15466,19600,0.05,test_train_45_02_05.png,15


/app/data_model/output


In [56]:
pr_df_list = []
ap_df_list = []

for checkpoint in checkpoint_numbers:
    
    pred_df_checkpoint = pred_df.loc[pred_df['checkpoint'] == checkpoint]
    
    # Set up the metrics instance
    metrics = DetectionMetrics(
    true_df=true_df,
    pred_df=pred_df_checkpoint,
    file_col=file_col,
    label_col=label_col,
    bbox_col=bbox_col,
    score_col=score_col)
    
    # Classify the predictions
    classifications, missed = metrics.classify_predictions_df(iou_threshold=0.5)
    pr_df = metrics.ap_from_classifications(classifications=classifications)
    pr_df_list.append(pr_df)
    
    # Get the AUC values
    ap_df = pr_df[[label_col, 'iou_threshold', 'auc', file_col]].\
        groupby([label_col, 'iou_threshold', 'auc']).nunique().\
        reset_index(drop=False).\
        rename(columns={file_col: 'n_predictions'}).\
        sort_values(by=label_col, ascending=True).\
        reset_index(drop=True)
    ap_df.insert(loc=0, column='model', value=model_name)
    ap_df.insert(loc=1, column='dataset', value=dataset_name)
    ap_df.insert(loc=2, column='checkpoint', value=checkpoint)
    ap_df.insert(loc=3, column='score_threshold', value=threshold)
    print(f'mAP for checkpoint {checkpoint} at IoU({iou_threshold}): {ap_df['auc'].mean(): .3f}')
    ap_df_list.append(ap_df)

# Save the output
precision_recall_df = pd.concat(pr_df_list, axis=0, ignore_index=True)
average_precision_df = pd.concat(ap_df_list, axis=0, ignore_index=True)

pr_data_name = f'{model_name}_predictions.parquet'
pr_data_file = os.path.join(output_dir, pr_data_name)
precision_recall_df.to_parquet(pr_data_file)
print(f'Saved Precision-Recall data: {pr_data_file}')

ap_data_name = f'{model_name}_ap_per_label.parquet'
ap_data_file = os.path.join(output_dir, ap_data_name)
average_precision_df.to_parquet(ap_data_file)
print(f'Saved Average Precision data: {ap_data_file}')

display(precision_recall_df.head(2))
display(average_precision_df.head(2))

mAP for checkpoint 19600 at IoU(0.5):  0.649
mAP for checkpoint 119900 at IoU(0.5):  0.793
mAP for checkpoint 120000 at IoU(0.5):  0.789
mAP for checkpoint 119950 at IoU(0.5):  0.790
mAP for checkpoint 119850 at IoU(0.5):  0.791
Saved Precision-Recall data: /app/data_model/output/rtdetr_dtx_251012_08_predictions.parquet
Saved Average Precision data: /app/data_model/output/rtdetr_dtx_251012_08_ap_per_label.parquet


,file_name,iou_threshold,score,bbox,ada,TP,IoU,n_missed,duplicate_TP,precision,recall,precision_label,recall_label,auc
0,test_train_226_14_05.png,0.5,0.313148,"[82, 0, 256, 231]",1,1,0.907051,0,False,1.0,0.083333,0.150943,2.0,0.757657
1,test_train_588_14_05.png,0.5,0.290197,"[0, 0, 173, 229]",1,1,0.673031,0,False,1.0,0.166667,0.150943,2.0,0.757657


,model,dataset,checkpoint,score_threshold,ada,iou_threshold,auc,n_predictions
0,rtdetr_dtx_251012_08,dataset_object_dentex_251011,19600,0.05,1,0.5,0.757657,78
1,rtdetr_dtx_251012_08,dataset_object_dentex_251011,19600,0.05,2,0.5,0.262328,70
